# PPE Detection — YOLOv8m Eğitim

# 1. Kurulum ve GPU Kontrolü

In [ ]:
# ultralytics ve roboflow kütüphanelerinin kurulumu
!pip install ultralytics roboflow pyyaml -q

import torch

gpu_durumu = torch.cuda.is_available()
print("-> GPU Aktifliği:", gpu_durumu)

if gpu_durumu:
    print("-> Kullanılan GPU:", torch.cuda.get_device_name(0))

# 2. Şahsi Veri Setinin İçe Aktarılması

In [ ]:
# roboflow üzerinden hazırlanan temiz ve optimize edilmiş veri seti çekiliyor
from roboflow import Roboflow

rf = Roboflow(api_key="apcl9rl23IZ4qbWDuLzV")
project = rf.workspace("tunahans-workspace-9glcc").project("cv-ppe-detection-npbbf")

version = project.version(1) 
dataset = version.download("yolov8")

print("-> Veri seti başarıyla indirildi.")
print("-> PATH:", dataset.location)

# 3. YOLO Eğitimi 

In [ ]:
from ultralytics import YOLO

# medium model hız/doğruluk dengesi için ilk etapta ideal.
model = YOLO("yolov8m.pt")

print("-> Eğitim başlıyor...")

# eğitim parametre config'i
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=32, # çift GPU nedeniyle 16 yerine 32
    project="/kaggle/working/results",
    name="yolov8m_custom_ppe",
    save=True,
    device=[0, 1] # -> çift GPU
)

print("-> Eğitim tamamlandı! Ağırlıklar results klasörüne eklendi.")